In [1]:
import os
os.chdir('..')
print(os.getcwd())

d:\pythonProject\IC Lab\Gait_analysis\pyskl


In [2]:
import re, gc
import numpy as np
import subprocess
from colab.tools import prinfo, config
from colab.PSO import PSO
import torch

In [3]:
def inference_split9(SSO_searcher, run):
    best_split8_log = SSO_searcher.ckpt[SSO_searcher.genBest][SSO_searcher.gBest]['message']['log']
    best_split8_log = os.path.basename(best_split8_log)
    command = f"python -m colab.training_tools.inference_split9 --txt {best_split8_log}"

    try:
        # 執行並收集輸出
        b = !{command}
        # print(b)
        
        # 使用 `prinfo()` 解析結果
        (
            train_cost, train_time, test_time, val_acc_4c, val_acc_3c, val_acc_2c, 
            test_acc_4c, test_acc_3c, test_acc_2c, f1, pre, rec, log
        ) = prinfo(b)
        
        print(f"Run : {run-1:>3} | Train_cost : {train_cost:>8.3f}s | Train_time : {train_time:>8.7f}s | "
                f"Test_time : {test_time:>8.3f}s | Val_acc_4c : {val_acc_4c:>10.7f} | "
                f"Val_acc_3c : {val_acc_3c:>10.7f} | Val_acc_2c : {val_acc_2c:>10.7f} | "
                f"Test_acc_4c : {test_acc_4c:>10.7f} | Test_acc_3c : {test_acc_3c:>10.7f} | "
                f"Test_acc_2c : {test_acc_2c:>10.7f} | F1 : {f1:>10.7f} | "
                f"Precision : {pre:>10.7f} | Recall : {rec:>10.7f} | Log : {log}")

        record_message = {
            'train_cost': train_cost,
            'train_time': train_time,
            'test_time': test_time,
            'val_acc_4c': val_acc_4c,
            'val_acc_3c': val_acc_3c,
            'val_acc_2c': val_acc_2c,
            'test_acc_4c': test_acc_4c,
            'test_acc_3c': test_acc_3c,
            'test_acc_2c': test_acc_2c,
            'f1': f1,
            'precision': pre,
            'recall': rec,
            'log': log
        }

        # 重新存split 9

        import uuid
        import pickle
        split = 9
        SSO_searcher.ckpt[SSO_searcher.genBest][SSO_searcher.gBest]['message'] = record_message
        unique_id = uuid.uuid4().hex[:8]  # 唯一識別代號，避免有搜尋結果相同的問題
        save_name = f"PSO_{'drunk'}_{9}_run{run-1}_Triplet"
        save_path = f"sso_result/{save_name}_ggen{SSO_searcher.genBest}_gsol{SSO_searcher.gBest}_searchtime{SSO_searcher.search_time}_{unique_id}.pkl"
        
        # 確保目錄存在
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        with open(save_path, 'wb') as f:
            pickle.dump(SSO_searcher.ckpt, f)
        print(f"📄 記錄檔已儲存至：{save_path}")
        print("-" * 40)

        # 打印輸出
        SSO_searcher.result_summary()

    except Exception as e:
        print(f"❌ 發生錯誤: {e}")
        # 確保所有變數有值，避免 `NoneType` 出錯
        train_cost = train_time = test_time = 0
        val_acc_4c = val_acc_3c = val_acc_2c = 0
        test_acc_4c = test_acc_3c = test_acc_2c = 0
        f1 = pre = rec = 0
        log = ""


In [ ]:
split = 8

params_range = {
'num_init'      : (1, 4),
'base_channel'  : (64, 256),
'stride_init'   : (1, 3),
'tkernel_init'  : (1, 3),               # *2+1

'act'           : (0, 3),               # activate function  
'opt'           : (0, 2),               # optimizer

'dropout_bk'    : (0.15, 0.30),         # dropout in block *10-2
'dropout_fc'    : (0.15, 0.30),         # dropout in fc *10-2

'lr'            : (0.001, 0.1),         # learning rate *10-3
'weight_decay'  : (0, 0.1),             # weight_decay *10-4
'momentum'      : (0.50, 0.99),         # momentum * 10-2
'margin'        : (0.20, 1.0),          # margin in Triplet Loss *10-2
'lambda_val'    : (0.0, 0.9),           # The ratio of Triplet Loss compare to Cross Entropy *10-2
}

# 設定初始解
params = {
    'data'          : 'drunk',          # 我自己讀特定資料集的方式
    'batch'         : 128,              # 可以調整~~
    'epoch'         : 80,               # 可以調整~~
    'feature'       : 'j',              # 我自己讀特定資料集的方式
    'split'         : split,
    
    'num_init'      : 4,
    'base_channel'  : 64,                # output_channel in init_layer
    'stride_init'   : 1,
    'tkernel_init'  : 1,

    'num_in'        : 0,                # num_layers_input_stream 固定捨棄後面兩個stream
    'num_main'      : 0,                # num_layers_main_stream 固定捨棄後面兩個stream

    'act'           : 0,                # activate function
    'opt'           : 0,                # optimizer
    
    'dropout_bk'    : 0,                # dropout in block
    'dropout_fc'    : 0,                # dropout in fc

    'lr'            : 0.1,              # learning rate
    'weight_decay'  : 0.0005,                # weight_decay
    'momentum'      : 0.9,               # momentum
    'margin'        : 0.6,              # margin in Triplet Loss
    'lambda_val'    : 0.5,              # The ratio of Triplet Loss compare to Cross Entropy
}
                                                                                                        
# 不用動
def get_param(X, keys=params_range.keys(), params=params):
    """
    將搜尋空間中編碼過的數值 X 映射回實際的模型參數設定，統一保留4位小數。

    Parameters:
        X (np.ndarray): 搜尋演算法產出的參數值列表。
        keys (list): 與 X 對應的參數名稱。
        params (dict): 預設參數字典。

    Returns:
        dict: 還原為實際值後的參數字典。
    """
    param = params.copy()

    for i, key in enumerate(keys):
        val = X[i]
        boundary = params_range[key]

        if key in ['num_init', 'base_channel', 'stride_init', 'act', 'opt']:
            param[key] = int(round(val))  # 直接四捨五入取整數
        elif key == 'tkernel_init':
            param[key] = int(round(val)) * 2 + 1  # 先取整數再 *2+1
        else:
            param[key] = val  # 其他保留小數
        # elif key == 'lr':
        #     param[key] = round(val * 1e-3, 4)
        # elif key == 'weight_decay':
        #     param[key] = round(val * 1e-4, 4)
        # elif key == 'momentum':
        #     param[key] = round(val * 1e-2, 4)
        # elif key == 'margin':
        #     param[key] = round(val * 1e-2, 4)
        # elif key == 'lambda_val':
        #     param[key] = round(val * 1e-2, 4)
        # elif (
        #     isinstance(boundary, (tuple, list)) and
        #     all(isinstance(b, int) for b in boundary)
        # ):
        #     param[key] = int(val)
        # else:
        #     param[key] = val
    return param

def fitness(X, run=None):
    # **釋放 GPU 記憶體**
    torch.cuda.empty_cache()
    
    param = get_param(X=X)
    param_args = " ".join([f"--{key} {str(value).strip()}" for key, value in param.items()])
    command = f"python -m colab.training_tools.kaggle_train_drunk_aug_multimetric_weight_small_lambda_gpu --mt {param_args}"
    
    try:
        # 使用 `!{command}` 在 Jupyter Notebook / Colab 內執行
        output = !{command}
        # print(output)
        # 解析輸出
        train_cost, train_time, test_time, val_acc_4c, val_acc_3c, val_acc_2c, \
        test_acc_4c, test_acc_3c, test_acc_2c, f1, pre, rec, log = prinfo(output)
        
        if run is not None:
            print(f"Run : {run:>3} | Train_cost : {train_cost:>8.3f}s | Train_time : {train_time:>8.7f}s | "
                f"Test_time : {test_time:>8.3f}s | Val_acc_4c : {val_acc_4c:>10.7f} | "
                f"Val_acc_3c : {val_acc_3c:>10.7f} | Val_acc_2c : {val_acc_2c:>10.7f} | "
                f"Test_acc_4c : {test_acc_4c:>10.7f} | Test_acc_3c : {test_acc_3c:>10.7f} | "
                f"Test_acc_2c : {test_acc_2c:>10.7f} | F1 : {f1:>10.7f} | "
                f"Precision : {pre:>10.7f} | Recall : {rec:>10.7f} | Log : {log}")

    except Exception as e:
        print(f"❌ 發生錯誤: {e}")
        # 確保所有變數有值，避免 `NoneType` 出錯
        train_cost = train_time = test_time = 0
        val_acc_4c = val_acc_3c = val_acc_2c = 0
        test_acc_4c = test_acc_3c = test_acc_2c = 0
        f1 = pre = rec = 0
        log = ""

    # 確保 `NoneType` 不影響計算
    fitness_value = (val_acc_4c or 0)

    # 確保 `log` 不為 `None`
    log = log if log is not None else ""

    record_message = {
        'train_cost': train_cost,
        'train_time': train_time,
        'test_time': test_time,
        'val_acc_4c': val_acc_4c,
        'val_acc_3c': val_acc_3c,
        'val_acc_2c': val_acc_2c,
        'test_acc_4c': test_acc_4c,
        'test_acc_3c': test_acc_3c,
        'test_acc_2c': test_acc_2c,
        'f1': f1,
        'precision': pre,
        'recall': rec,
        'log': log
    }
    return fitness_value, record_message

for run in range(1, 30):
    save_name = f"PSO_{params['data']}_{params['split']}_run{run-1}_Triplet"
    PSO_searcher = PSO(
        Ngen=10,                                # 世代數
        Nsol=10,                                # 解的數量
        w=0.8,                                  # 全局最佳解的權重
        c1=1.2,                                 # 個體最佳解的權重
        c2=0.8,                                 # 隨機解的權重
        save_name=save_name,                    # 記錄檔儲存前綴
        fitness=fitness,                        # 適應度函數
        base_param=params,                      # 基礎參數 (可選)
        boundary=params_range,                  # 支持 tuple 或 list (詳細說明請參照前述)
        direction="maximize",                   # 優化方向 ('maximize', 'minimize')
        )
    print("split 8")
    # 直接運行
    PSO_searcher.run()

    print("split 9")
    inference_split9(PSO_searcher, run=run)

split 8
